# Analysis of HEPMC3 Files and Smearing Behaviour

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import uproot
import glob
import awkward as ak
import itertools
import yaml
import os
import sys
from tqdm import tqdm
from pathlib import Path
import atlasify as atl
atl.ATLAS = "ColliderML"

sys.path.append("../")
import pyhepmc as hep
from pyhepmc.io import WriterAscii

## HEPMC Merging and Smearing

### Debug

In [26]:
def smear_vertex_position(event, vertex_sigmas):
    """Apply Gaussian smearing to all vertices in an event"""
    # Generate one smearing offset for the whole event
    offset = np.array([
        np.random.normal(0, vertex_sigmas['xy']),  # mm
        np.random.normal(0, vertex_sigmas['xy']),  # mm
        np.random.normal(0, vertex_sigmas['z']),   # mm
        np.random.normal(0, vertex_sigmas['t'])    # ns (time in ns)
    ])
    
    # Create new position vector for each vertex
    for vertex in event.vertices:
        new_pos = hep.FourVector(
            vertex.position.x + offset[0],
            vertex.position.y + offset[1],
            vertex.position.z + offset[2],
            vertex.position.t + offset[3]
        )
        vertex.position = new_pos
    
    return event

def merge_events(signal_event, pileup_events, vertex_sigmas, logger):
    """Merge signal and multiple pileup events into a single event with vertex smearing"""
    # Create new event with same units as input
    merged = hep.GenEvent(hep.Units.GEV, hep.Units.MM)
    
    # First add signal event with smearing
    signal_event = smear_vertex_position(signal_event, vertex_sigmas)
    
    # Create new signal vertices
    sig_vertices = []
    for vertex in signal_event.vertices:
        v1 = hep.GenVertex(vertex.position)
        sig_vertices.append(v1)
    
    # Add signal particles and connect them to vertices
    for particle in signal_event.particles:
        p1 = hep.GenParticle(
            particle.momentum,
            particle.pid,
            particle.status
        )
        p1.generated_mass = particle.generated_mass
        
        # Handle production vertex
        if particle.production_vertex.id < 0:
            production_vertex = particle.production_vertex.id
            sig_vertices[abs(production_vertex)-1].add_particle_out(p1)
            merged.add_particle(p1)
        else:
            merged.add_particle(p1)
        
        # Handle end vertex if it exists
        if particle.end_vertex:
            end_vertex = particle.end_vertex.id
            sig_vertices[abs(end_vertex)-1].add_particle_in(p1)
    
    # Add all signal vertices
    for vertex in sig_vertices:
        merged.add_vertex(vertex)
    
    # Now add pileup events with smearing
    for pileup_event in pileup_events:
        pileup_event = smear_vertex_position(pileup_event, vertex_sigmas)
        
        # Create new pileup vertices
        pileup_vertices = []
        for vertex in pileup_event.vertices:
            v1 = hep.GenVertex(vertex.position)
            pileup_vertices.append(v1)
        
        # Add pileup particles and connect them to vertices
        for particle in pileup_event.particles:
            p1 = hep.GenParticle(
                particle.momentum,
                particle.pid,
                particle.status
            )
            p1.generated_mass = particle.generated_mass
            
            if particle.production_vertex.id < 0:
                production_vertex = particle.production_vertex.id
                pileup_vertices[abs(production_vertex)-1].add_particle_out(p1)
                merged.add_particle(p1)
            else:
                merged.add_particle(p1)
            
            if particle.end_vertex:
                end_vertex = particle.end_vertex.id
                pileup_vertices[abs(end_vertex)-1].add_particle_in(p1)
        
        # Add all pileup vertices
        for vertex in pileup_vertices:
            merged.add_vertex(vertex)
    
    return merged

def merge_hepmc_files(signal_path, pileup_path, output_path, vertex_sigmas, logger=None):
    """Merge signal and pileup HepMC3 files into a single file with vertex smearing"""
    # logger = logger or setup_logging("MergeHepMC")
    # logger.info(f"Merging HepMC3 files:")
    # logger.info(f"Signal: {signal_path}")
    # logger.info(f"Pileup: {pileup_path}")
    # logger.info(f"Output: {output_path}")
    
    # Load all signal events
    signal_events = []
    with hep.open(signal_path) as f:
        for event in f:
            signal_events.append(event)
    
    # Load all pileup events
    pileup_events = []
    with hep.open(pileup_path) as f:
        for event in f:
            pileup_events.append(event)
    
    # Calculate pileup events per signal event
    n_pileup_per_signal = len(pileup_events) // len(signal_events)
    # logger.info(f"Found {len(signal_events)} signal events with {n_pileup_per_signal} pileup events each")
    
    # Write merged events
    with WriterAscii(str(output_path)) as f:
        for i, signal_event in enumerate(signal_events):
            # Get corresponding pileup events for this signal event
            start_idx = i * n_pileup_per_signal
            end_idx = start_idx + n_pileup_per_signal
            event_pileup = pileup_events[start_idx:end_idx]
            
            # Merge events with vertex smearing
            merged_event = merge_events(signal_event, event_pileup, vertex_sigmas, logger)
            merged_event.event_number = i
            
            # Write merged event
            f.write_event(merged_event)
    
    # logger.info("Merge complete")

In [27]:
signal_path = "/global/cfs/projectdirs/m4958/usr/danieltm/ColliderML/software/colliderml_dev/scripts/generation/madgraph_outputs/TTbar_NLO_FXFX_10kEvents_Test/events_PYTHIA8_0.hepmc.gz"
pileup_path = "/global/cfs/projectdirs/m4958/data/ColliderML/outputs/madgraph_merge_testing/pileup_only/merged_events.hepmc3"

In [15]:
# Load signal events
with hep.open(signal_path) as f:
    for event in tqdm(f):
        signal_event = event
        break

with hep.open(pileup_path) as f:
    for event in tqdm(f):
        pileup_event = event
        break

0it [00:00, ?it/s]
0it [00:00, ?it/s]


In [16]:
len(signal_event.particles), len(signal_event.vertices), len(pileup_event.particles), len(pileup_event.vertices)

(785, 433, 9546, 5541)

In [19]:
vertex_sigmas = {
            'xy': 0.0125,                    # mm
            'z': 55.5,                      # mm
            't': 5.0               # Should be converted to light-mm!!
        }

In [20]:
"""Merge signal and multiple pileup events into a single event with vertex smearing"""
# Create new event with same units as input
merged = hep.GenEvent(hep.Units.GEV, hep.Units.MM)

# First add signal event with smearing
signal_event = smear_vertex_position(signal_event, vertex_sigmas)

# Create new signal vertices
sig_vertices = []
for vertex in signal_event.vertices:
    v1 = hep.GenVertex(vertex.position)
    sig_vertices.append(v1)

# Add signal particles and connect them to vertices
for particle in signal_event.particles:
    p1 = hep.GenParticle(
        particle.momentum,
        particle.pid,
        particle.status
    )
    p1.generated_mass = particle.generated_mass
    
    # Handle production vertex
    if particle.production_vertex.id < 0:
        production_vertex = particle.production_vertex.id
        sig_vertices[abs(production_vertex)-1].add_particle_out(p1)
        merged.add_particle(p1)
    else:
        merged.add_particle(p1)
    
    # Handle end vertex if it exists
    if particle.end_vertex:
        end_vertex = particle.end_vertex.id
        sig_vertices[abs(end_vertex)-1].add_particle_in(p1)

# Add all signal vertices
for vertex in sig_vertices:
    merged.add_vertex(vertex)

In [22]:
# Now add pileup events with smearing

pileup_event = smear_vertex_position(pileup_event, vertex_sigmas)

# Create new pileup vertices
pileup_vertices = []
for vertex in pileup_event.vertices:
    v1 = hep.GenVertex(vertex.position)
    pileup_vertices.append(v1)

# Add pileup particles and connect them to vertices
for particle in pileup_event.particles:
    p1 = hep.GenParticle(
        particle.momentum,
        particle.pid,
        particle.status
    )
    p1.generated_mass = particle.generated_mass
    
    if particle.production_vertex.id < 0:
        production_vertex = particle.production_vertex.id
        pileup_vertices[abs(production_vertex)-1].add_particle_out(p1)
        merged.add_particle(p1)
    else:
        merged.add_particle(p1)
    
    if particle.end_vertex:
        end_vertex = particle.end_vertex.id
        pileup_vertices[abs(end_vertex)-1].add_particle_in(p1)

# Add all pileup vertices
for vertex in pileup_vertices:
    merged.add_vertex(vertex)

In [23]:
with WriterAscii(str("merged_events.hepmc3")) as f:
    
    # Get corresponding pileup events for this signal event
    merged.event_number = 0
    
    # Write merged event
    f.write_event(merged)

### Final Version

In [31]:
def merge_events(signal_event, pileup_event, vertex_sigmas, logger):
    """Merge signal and multiple pileup events into a single event with vertex smearing"""
    # Create new event with same units as input
    merged = hep.GenEvent(hep.Units.GEV, hep.Units.MM)
    
    # First add signal event with smearing
    signal_event = smear_vertex_position(signal_event, vertex_sigmas)
    
    # Create new signal vertices
    sig_vertices = []
    for vertex in signal_event.vertices:
        v1 = hep.GenVertex(vertex.position)
        sig_vertices.append(v1)
    
    # Add signal particles and connect them to vertices
    for particle in signal_event.particles:
        p1 = hep.GenParticle(
            particle.momentum,
            particle.pid,
            particle.status
        )
        p1.generated_mass = particle.generated_mass
        
        # Handle production vertex
        if particle.production_vertex.id < 0:
            production_vertex = particle.production_vertex.id
            sig_vertices[abs(production_vertex)-1].add_particle_out(p1)
            merged.add_particle(p1)
        else:
            merged.add_particle(p1)
        
        # Handle end vertex if it exists
        if particle.end_vertex:
            end_vertex = particle.end_vertex.id
            sig_vertices[abs(end_vertex)-1].add_particle_in(p1)
    
    # Add all signal vertices
    for vertex in sig_vertices:
        merged.add_vertex(vertex)
    
    # Now add pileup events with smearing
    pileup_event = smear_vertex_position(pileup_event, vertex_sigmas)
    
    # Create new pileup vertices
    pileup_vertices = []
    for vertex in pileup_event.vertices:
        v1 = hep.GenVertex(vertex.position)
        pileup_vertices.append(v1)
    
    # Add pileup particles and connect them to vertices
    for particle in pileup_event.particles:
        p1 = hep.GenParticle(
            particle.momentum,
            particle.pid,
            particle.status
        )
        p1.generated_mass = particle.generated_mass
        
        if particle.production_vertex.id < 0:
            production_vertex = particle.production_vertex.id
            pileup_vertices[abs(production_vertex)-1].add_particle_out(p1)
            merged.add_particle(p1)
        else:
            merged.add_particle(p1)
        
        if particle.end_vertex:
            end_vertex = particle.end_vertex.id
            pileup_vertices[abs(end_vertex)-1].add_particle_in(p1)
    
    # Add all pileup vertices
    for vertex in pileup_vertices:
        merged.add_vertex(vertex)
    
    return merged
def merge_hepmc_files(signal_path, pileup_path, num_events, output_path, vertex_sigmas, logger=None):
    """Merge signal and pileup HepMC3 files into a single file with vertex smearing"""
    # logger = logger or setup_logging("MergeHepMC")
    # logger.info(f"Merging HepMC3 files:")
    # logger.info(f"Signal: {signal_path}")
    # logger.info(f"Pileup: {pileup_path}")
    # logger.info(f"Output: {output_path}")
    
    # Load all signal events
    signal_events = []
    with hep.open(signal_path) as f:
        for i, event in enumerate(f):
            if i >= num_events:
                break
            signal_events.append(event)
    
    # Load all pileup events
    pileup_events = []
    with hep.open(pileup_path) as f:
        for i, event in enumerate(f):
            if i >= num_events:
                break
            pileup_events.append(event)

    if len(signal_events) != len(pileup_events):
        print(f"Warning: Number of signal events ({len(signal_events)}) does not match number of pileup events ({len(pileup_events)})")
        print(f"Truncating to min of {min(len(signal_events), len(pileup_events))}")
        signal_events = signal_events[:min(len(signal_events), len(pileup_events))]
        pileup_events = pileup_events[:min(len(signal_events), len(pileup_events))]
    
    # Calculate pileup events per signal event
    n_pileup_per_signal = len(pileup_events) // len(signal_events)
    # logger.info(f"Found {len(signal_events)} signal events with {n_pileup_per_signal} pileup events each")
    
    # Write merged events
    with WriterAscii(str(output_path)) as f:
        for i, (signal_event, pileup_event) in enumerate(zip(signal_events, pileup_events)):
            # Merge events with vertex smearing
            merged_event = merge_events(signal_event, pileup_event, vertex_sigmas, logger)
            merged_event.event_number = i
            
            # Write merged event
            f.write_event(merged_event)
    
    # logger.info("Merge complete")

In [32]:
merge_hepmc_files(signal_path, pileup_path, 10, "merged_events.hepmc3", vertex_sigmas)

## HepMC Splitting

In [5]:
input_path = "/global/cfs/cdirs/m4958/data/ColliderML/outputs/madgraph_merge_testing/run_0/events_PYTHIA8_0.hepmc.gz"
output_path = "/global/cfs/cdirs/m4958/data/ColliderML/outputs/madgraph_merge_testing/"
events_per_file = 10
current_f_out = None
file_path = None


In [7]:
with hep.open(input_path) as f_in:
    for i, event in tqdm(enumerate(f_in), total=10000):
        if i % events_per_file == 0:
            # If there's an open file, close it before opening a new one
            if current_f_out:
                current_f_out.close()
            
            file_idx = i // events_per_file
            current_output_dir = os.path.join(output_path, f"run_{file_idx}")
            os.makedirs(current_output_dir, exist_ok=True)
            file_path = os.path.join(current_output_dir, "events.hepmc")
            current_f_out = WriterAscii(file_path)
        
        if current_f_out:
            current_f_out.write_event(event)

        # break after 100 events for testing, remove for full run
        if i >= 999: # Process 1000 events (0 to 999) which is 100 files
            break

# Ensure the last file is closed after the loop
if current_f_out:
    current_f_out.close()

 10%|▉         | 999/10000 [00:05<00:49, 182.94it/s]
